# Episode 22: Long context LLM: stretching RoPE with YaRN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep022-long-context.ipynb)

We stretch the rotary embeddings of our web model to read four times its window, four ways and then with fifteen minutes of fine-tuning, and measure the loss by position and a pass key hidden in filler.

[The video](https://youtu.be/2C2JYv2agzo) · [The episode's page](https://mvgpt.nenuadrian.com/ep022-long-context/) · [The docs](https://nenuadrian.github.io/mvGPT/long-context/) · [The code at `ep022/end`](https://github.com/nenuadrian/mvGPT/tree/ep022/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep022/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("pretrain-web")

## The data
FineWeb-Edu's pages, for the long ones among them.

In [ ]:
!python data.py --web 1  # FineWeb-Edu: two files of about 2 GB, the web merges, the held-out pages and a file of tokens

## The tests
Each way's speeds, and a stretched model against the whole sequence.

In [ ]:
!python -m pytest tests/test_long.py -v

## Past the window
The loss by position past the 1,024 tokens the web model trained on: a sliding window, and
four ways to stretch RoPE. Then a pass key hidden in filler, by length.

In [ ]:
!python long.py checkpoints/pretrain-web/model.pt
!python passkey.py checkpoints/pretrain-web/model.pt

## Fine-tuning at 4,096 tokens
Fifteen minutes stretched with PI, then fifteen with YaRN.

In [ ]:
!python long_train.py pi 15
!python long_train.py yarn 15
# or, to skip the thirty minutes, the video's: fetch("long-pi", "long-yarn")

## After

In [ ]:
!python long.py checkpoints/long-pi/model.pt
!python long.py checkpoints/long-yarn/model.pt
!python passkey.py checkpoints/long-pi/model.pt
!python passkey.py checkpoints/long-yarn/model.pt

Next: [episode 23, Muon vs AdamW, Lion and Sophia: a fair race](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep023-optimizers.ipynb)